In [1]:
import sys
sys.path.append('../src')

In [2]:
from VCFParser import VCFParser

vcf_file = '../data/pgx.passed.vcf'
vcf_parser = VCFParser(vcf_file)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
#Test ensembl annotations
from QC_gate import requires_pass_qc
from ensembl import get_ensembl_info, parse_ensembl_response

with vcf_parser as parsed:
    for variant in parsed:
        ensembl_info = get_ensembl_info(variant)
        parsed_info = parse_ensembl_response(ensembl_info)

In [5]:
#Test pharmgkb annotations
from QC_gate import requires_pass_qc
from ensembl import get_ensembl_info, parse_ensembl_response
from pharmgkb import get_pharmgkb_info

#I just want to test on the first passed variant, so I will break after the first iteration
#There is rate limiting on the APIs, so I don't want to make too many calls during testing but variants are few, so I will get the first 10
with vcf_parser as parsed:
    counter = 0
    for variant in parsed:
        if counter >= 10:
            break
        requires_pass_qc(variant)
        ensembl_info = get_ensembl_info(variant)
        parsed_info = parse_ensembl_response(ensembl_info)
        pharmgkb_info = get_pharmgkb_info(variant, parsed_info)
        counter += 1
        break

#Check the PharmGKB information
print(pharmgkb_info)

{'chr1:119507310:C/A': []}


In [4]:
from pharmgkb import get_pharmgkb_info

#Code above didn't do as I expected, so I will force test with mock data of a known variant that is in PharmGKB - ACE
variant = type('Variant', (object,), {'CHROM': 'chr17', 'POS': 61554422})()  # Mock variant with known chromosome and position
parsed_info = {'gene_symbol': 'ACE'}  # Mock parsed info with known gene
pharmgkb_info = get_pharmgkb_info(parsed_info)
print(pharmgkb_info)

[{'id': 1185012329, 'drug': ['tacrolimus'], 'isAssociated': False, 'significance': {'id': 769229667, 'resource': 'Association Significance', 'synonyms': [], 'term': 'no', 'termId': 'variantAnnotationSignificance:769229667', 'valid': True}, 'evidence': 15083695}, {'id': 1018613244, 'drug': ['quinapril'], 'isAssociated': True, 'significance': {'id': 769229666, 'resource': 'Association Significance', 'synonyms': [], 'term': 'yes', 'termId': 'variantAnnotationSignificance:769229666', 'valid': True}, 'evidence': 11394446}, {'id': 1183491887, 'drug': ['lisinopril'], 'isAssociated': True, 'significance': {'id': 769229667, 'resource': 'Association Significance', 'synonyms': [], 'term': 'no', 'termId': 'variantAnnotationSignificance:769229667', 'valid': True}, 'evidence': 878189}, {'id': 982044530, 'drug': ['captopril'], 'isAssociated': True, 'significance': {'id': 769229666, 'resource': 'Association Significance', 'synonyms': [], 'term': 'yes', 'termId': 'variantAnnotationSignificance:76922966

In [4]:
len(pharmgkb_info)

102

In [6]:
from interpreter import LLMInterpreter, build_content

content = build_content(variant, ensembl_info, pharmgkb_info)
print(content)  # verify the prompt looks right first


NameError: name 'ensembl_info' is not defined

In [5]:
import anthropic
client = anthropic.Anthropic()

gene = "HSD3B1"
consequence = "intronic_variant"

response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=2048,
    system="You are a helpful assistant. Always follow output format instructions exactly.",
    messages=[{
        "role": "user",
        "content": f"""Write one sentence about {gene}.

Then output this JSON block:
````json
{{
    "gene": "{gene}",
    "consequence": "{consequence}",
    "interpretation": ""
}}
```"""
    }]
)

print(response.content[0].text)

HSD3B1 encodes 3β-hydroxysteroid dehydrogenase type 1, an enzyme essential for the biosynthesis of steroid hormones including progesterone, androgens, and estrogens in peripheral tissues such as the placenta, skin, and breast.

```json
{
    "gene": "HSD3B1",
    "consequence": "intronic_variant",
    "interpretation": ""
}
```


In [4]:
print(structured)

{}
